In [ ]:
# %% 셀 1: 데이터 로드 (per-pixel slot ID 가능성 검증)
import os, json
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import Counter, defaultdict

POS_DIR = "./data/8_telop_position"
GRID_W = 80
GRID_H = 80
FPS = 10
NUM_WORKERS = 32


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        x0 = max(0, gx - gw // 2)
        y0 = max(0, gy - gh // 2)
        x1 = min(GRID_W, x0 + gw)
        y1 = min(GRID_H, y0 + gh)
        inst_list.append({
            "start": inst["start_sec"],
            "end":   inst["end_sec"],
            "x0": x0, "y0": y0, "x1": x1, "y1": y1,
            "text": inst["text"],
            "text_len": len(inst["text"]),
        })
    return {
        "channel": channel,
        "instances": inst_list,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        samples.append(fut.result())

print(f"\n✅ 총 {len(samples):,}개 샘플 / {sum(len(s['instances']) for s in samples):,}개 인스턴스")


In [ ]:
# %% 셀 2: 프레임 단위 픽셀 overlap 통계 (전체 frame, 병렬)
# 핵심 질문:
#   1) per-pixel slot ID 분류 가능한가? (overlap 비율 충분히 작은가)
#   2) 한 픽셀 최대 overlap (Top-N cap 결정)
#   3) MAX_ACTIVE_PER_FRAME=127 (K=127, 클래스=128)이 충분한가

K_TARGET = 127  # MAX_ACTIVE_PER_FRAME 후보 (BG 포함 128 클래스)
TOP_N    = 4    # per-pixel slot ID rank 수


def frame_stats_worker(s):
    duration = max(s["duration"], 0.1)
    n_frames = max(1, int(duration * FPS) + 1)
    instances = s["instances"]

    active_count_local  = Counter()
    pixel_overlap_local = Counter()
    max_overlap_local   = Counter()  # per-frame max overlap 히스토그램
    n_total = n_with_overlap = 0
    total_telop = total_overlap = 0
    n_exceed_K = 0   # K_TARGET 초과 frame 수

    if not instances:
        active_count_local[0] = n_frames
        return (active_count_local, pixel_overlap_local, max_overlap_local,
                n_frames, 0, 0, 0, 0)

    inst_starts = np.array([i["start"] for i in instances])
    inst_ends   = np.array([i["end"]   for i in instances])

    for fi in range(n_frames):
        t = fi / FPS
        active = (inst_starts <= t + 0.05) & (inst_ends > t)
        active_idx = np.where(active)[0]
        n_a = len(active_idx)
        active_count_local[n_a] += 1
        n_total += 1
        if n_a > K_TARGET:
            n_exceed_K += 1

        if n_a == 0:
            continue

        pixel_count = np.zeros((GRID_H, GRID_W), dtype=np.int16)
        for ai in active_idx:
            inst = instances[ai]
            pixel_count[inst["y0"]:inst["y1"], inst["x0"]:inst["x1"]] += 1

        max_ov = int(pixel_count.max())
        max_overlap_local[max_ov] += 1
        n_tel = int((pixel_count >= 1).sum())
        n_ov  = int((pixel_count >= 2).sum())
        total_telop   += n_tel
        total_overlap += n_ov
        if n_ov > 0:
            n_with_overlap += 1
        for cnt in range(1, max_ov + 1):
            pixel_overlap_local[cnt] += int((pixel_count == cnt).sum())

    return (active_count_local, pixel_overlap_local, max_overlap_local,
            n_total, n_with_overlap, total_telop, total_overlap, n_exceed_K)


active_count_dist     = Counter()
pixel_overlap_dist    = Counter()
max_overlap_hist      = Counter()
n_total_frames        = 0
n_frames_with_overlap = 0
total_telop_px        = 0
total_overlap_px      = 0
n_frames_exceed_K     = 0

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = [pool.submit(frame_stats_worker, s) for s in samples]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="frame stats"):
        ac, po, mo, nt, nw, tt, to, nx = fut.result()
        active_count_dist.update(ac)
        pixel_overlap_dist.update(po)
        max_overlap_hist.update(mo)
        n_total_frames        += nt
        n_frames_with_overlap += nw
        total_telop_px        += tt
        total_overlap_px      += to
        n_frames_exceed_K     += nx

print(f"\n분석 frame: {n_total_frames:,}")
print(f"Overlap 있는 frame: {n_frames_with_overlap:,} "
      f"({100*n_frames_with_overlap/max(n_total_frames,1):.2f}%)")
print(f"Telop 덮은 pixel 총합: {total_telop_px:,}")
print(f"Overlap pixel (≥2 bbox): {total_overlap_px:,} "
      f"({100*total_overlap_px/max(total_telop_px,1):.3f}%)")
print(f"Active slot > K_TARGET={K_TARGET} frame: {n_frames_exceed_K:,} "
      f"({100*n_frames_exceed_K/max(n_total_frames,1):.4f}%)")


In [ ]:
# %% 셀 3: Summary 출력

print("=" * 70)
print("Active slots per frame distribution")
print("=" * 70)
total = sum(active_count_dist.values())
cum = 0
shown_99   = False
shown_999  = False
shown_9999 = False
for n in sorted(active_count_dist.keys()):
    c = active_count_dist[n]
    cum += c
    pct = c / total * 100
    cum_pct = cum / total * 100
    if pct >= 0.01 or (not shown_9999 and cum_pct >= 99.99) or n <= 20:
        marker = ""
        if not shown_99 and cum_pct >= 99.0:
            marker, shown_99 = " <- 99%", True
        elif not shown_999 and cum_pct >= 99.9:
            marker, shown_999 = " <- 99.9%", True
        elif not shown_9999 and cum_pct >= 99.99:
            marker, shown_9999 = " <- 99.99%", True
        print(f"  active={n:3d}: {c:>10,}  ({pct:5.2f}%)  [cum {cum_pct:6.2f}%]{marker}")

max_active = max(active_count_dist.keys()) if active_count_dist else 0
print(f"\n  최대 active slot/frame: {max_active}")

print()
print("=" * 70)
print("Pixel coverage by # bboxes")
print("=" * 70)
total_px = sum(pixel_overlap_dist.values())
cum_px = 0
for cnt in sorted(pixel_overlap_dist.keys()):
    px = pixel_overlap_dist[cnt]
    pct = px / max(total_px, 1) * 100
    cum_px += px
    cum_pct = cum_px / max(total_px, 1) * 100
    bar = "█" * min(int(pct / 2), 50)
    print(f"  {cnt:2d} bbox cover: {px:>14,} px ({pct:7.4f}%) [cum {cum_pct:7.4f}%]  {bar}")

print()
print("=" * 70)
print("Per-frame max overlap (한 frame에 가장 많이 겹친 pixel의 bbox 수)")
print("=" * 70)
total_f = sum(max_overlap_hist.values())
cum_f = 0
# percentile lookup
def percentile_from_hist(hist, q):
    target = total_f * q / 100
    c = 0
    for v in sorted(hist.keys()):
        c += hist[v]
        if c >= target:
            return v
    return max(hist.keys())

for q in [50, 75, 90, 95, 99, 99.5, 99.9, 99.99]:
    val = percentile_from_hist(max_overlap_hist, q)
    print(f"  {q:5.2f}th percentile: {val}")
print(f"  Max:                {max(max_overlap_hist.keys())}")
mean_max = sum(k*v for k,v in max_overlap_hist.items()) / max(total_f, 1)
print(f"  Mean:               {mean_max:.3f}")


In [ ]:
# %% 셀 4: 채널별 overlap 정도 (전체 frame, 병렬)

def channel_stats_worker(s):
    duration = max(s["duration"], 0.1)
    n_frames = max(1, int(duration * FPS) + 1)
    instances = s["instances"]
    out = {
        "channel": s["channel"],
        "frames": 0, "overlap_frames": 0,
        "max_active": 0, "sum_active": 0,
        "max_overlap_px": 0,
    }
    if not instances:
        out["frames"] = n_frames
        return out

    inst_starts = np.array([i["start"] for i in instances])
    inst_ends   = np.array([i["end"]   for i in instances])

    for fi in range(n_frames):
        t = fi / FPS
        active = (inst_starts <= t + 0.05) & (inst_ends > t)
        active_idx = np.where(active)[0]
        n_a = len(active_idx)
        out["frames"]      += 1
        out["sum_active"]  += n_a
        if n_a > out["max_active"]:
            out["max_active"] = n_a

        if n_a < 2:
            continue
        pixel_count = np.zeros((GRID_H, GRID_W), dtype=np.int16)
        for ai in active_idx:
            inst = instances[ai]
            pixel_count[inst["y0"]:inst["y1"], inst["x0"]:inst["x1"]] += 1
        if (pixel_count >= 2).any():
            out["overlap_frames"] += 1
        m = int(pixel_count.max())
        if m > out["max_overlap_px"]:
            out["max_overlap_px"] = m
    return out


ch_stats = defaultdict(lambda: {
    "frames": 0, "overlap_frames": 0,
    "max_active": 0, "sum_active": 0,
    "max_overlap_px": 0,
})

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = [pool.submit(channel_stats_worker, s) for s in samples]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="per-channel"):
        r = fut.result()
        ch = r["channel"]
        ch_stats[ch]["frames"]         += r["frames"]
        ch_stats[ch]["overlap_frames"] += r["overlap_frames"]
        ch_stats[ch]["sum_active"]     += r["sum_active"]
        ch_stats[ch]["max_active"]      = max(ch_stats[ch]["max_active"], r["max_active"])
        ch_stats[ch]["max_overlap_px"]  = max(ch_stats[ch]["max_overlap_px"], r["max_overlap_px"])

ch_list = []
for ch, st in ch_stats.items():
    if st["frames"] > 0:
        ch_list.append((
            ch,
            st["overlap_frames"] / st["frames"] * 100,
            st["max_active"],
            st["sum_active"] / st["frames"],
            st["max_overlap_px"],
        ))

ch_list.sort(key=lambda x: -x[1])

print(f"총 {len(ch_list)}개 채널 / 평균 overlap rate: {np.mean([x[1] for x in ch_list]):.2f}%")
print()
print(f"{'Channel':<35} {'Overlap%':>10} {'MaxAct':>8} {'AvgAct':>8} {'MaxOvPx':>8}")
print("─" * 75)
print("[TOP 15 by overlap rate]")
for ch, ov, ma, aa, mp in ch_list[:15]:
    print(f"{ch[:33]:<35} {ov:>10.2f} {ma:>8d} {aa:>8.2f} {mp:>8d}")

print()
print("[TOP 5 by max_active]")
for ch, ov, ma, aa, mp in sorted(ch_list, key=lambda x: -x[2])[:5]:
    print(f"{ch[:33]:<35} {ov:>10.2f} {ma:>8d} {aa:>8.2f} {mp:>8d}")

print()
print("[BOTTOM 5 by overlap rate]")
for ch, ov, ma, aa, mp in ch_list[-5:]:
    print(f"{ch[:33]:<35} {ov:>10.2f} {ma:>8d} {aa:>8.2f} {mp:>8d}")


In [ ]:
# %% 셀 5: Worst overlap frame 시각화 (per-pixel slot ID 어려움 정도 직관적으로 확인)
import matplotlib.pyplot as plt

worst_frames = []
for s in samples:
    duration = max(s["duration"], 0.1)
    n_frames = max(1, int(duration * FPS) + 1)
    instances = s["instances"]
    if not instances:
        continue
    inst_starts = np.array([i["start"] for i in instances])
    inst_ends   = np.array([i["end"]   for i in instances])

    for fi in range(n_frames):
        t = fi / FPS
        active = (inst_starts <= t + 0.05) & (inst_ends > t)
        active_idx = np.where(active)[0]
        n_a = len(active_idx)
        if n_a < 3:
            continue

        pixel_count = np.zeros((GRID_H, GRID_W), dtype=np.int16)
        for ai in active_idx:
            inst = instances[ai]
            pixel_count[inst["y0"]:inst["y1"], inst["x0"]:inst["x1"]] += 1

        max_ov = int(pixel_count.max())
        if max_ov >= 4:  # TOP_N 초과 후보를 우선 수집
            worst_frames.append((max_ov, s["channel"], pixel_count.copy(), n_a, t))
            if len(worst_frames) >= 200:
                break
    if len(worst_frames) >= 200:
        break

# Top-N(4) 초과 사례가 부족하면 max_ov >= 3 으로 보충
if len(worst_frames) < 12:
    for s in samples:
        duration = max(s["duration"], 0.1)
        n_frames = max(1, int(duration * FPS) + 1)
        instances = s["instances"]
        if not instances:
            continue
        inst_starts = np.array([i["start"] for i in instances])
        inst_ends   = np.array([i["end"]   for i in instances])
        for fi in range(n_frames):
            t = fi / FPS
            active = (inst_starts <= t + 0.05) & (inst_ends > t)
            active_idx = np.where(active)[0]
            if len(active_idx) < 3:
                continue
            pixel_count = np.zeros((GRID_H, GRID_W), dtype=np.int16)
            for ai in active_idx:
                inst = instances[ai]
                pixel_count[inst["y0"]:inst["y1"], inst["x0"]:inst["x1"]] += 1
            max_ov = int(pixel_count.max())
            if max_ov >= 3:
                worst_frames.append((max_ov, s["channel"], pixel_count.copy(), len(active_idx), t))
                if len(worst_frames) >= 200:
                    break
        if len(worst_frames) >= 200:
            break

worst_frames.sort(key=lambda x: -x[0])

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for idx, ax in enumerate(axes.flat):
    if idx >= len(worst_frames):
        ax.axis("off")
        continue
    max_ov, ch, pc, n_a, t = worst_frames[idx]
    im = ax.imshow(pc, cmap="hot", vmin=0, vmax=max(max_ov, 1))
    ax.set_title(f"{ch[:25]}\nt={t:.1f}s active={n_a} maxov={max_ov}", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.04)
plt.suptitle("Pixel overlap heatmaps (max overlap ≥ 3)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# %% 셀 6: 결정값 검증 (MAX_ACTIVE_PER_FRAME=127, TOP_N=4)

print("=" * 70)
print(f"검증 대상: MAX_ACTIVE_PER_FRAME = K = {K_TARGET}  (클래스 = 1+K = {K_TARGET+1})")
print(f"          per-pixel TOP_N      = {TOP_N}")
print("=" * 70)

# (1) K=127 충분한가: active slot/frame > K 인 frame 비율
total_f = sum(active_count_dist.values())
exceed = sum(c for k, c in active_count_dist.items() if k > K_TARGET)
exceed_pct = 100 * exceed / max(total_f, 1)
print(f"\n[1] active slot > {K_TARGET} 인 frame: {exceed:,} / {total_f:,}  ({exceed_pct:.5f}%)")
if exceed_pct < 0.01:
    print(f"    OK — 99.99% 이상의 frame이 K={K_TARGET}로 커버됨")
elif exceed_pct < 0.1:
    print(f"    경고 — overflow {exceed_pct:.3f}%, 잘리는 슬롯 처리 정책 필요")
else:
    print(f"    부족 — K 증가 검토 필요")

# active slot 99/99.9/99.99 percentile
def percentile_active(q):
    target = total_f * q / 100
    c = 0
    for k in sorted(active_count_dist.keys()):
        c += active_count_dist[k]
        if c >= target:
            return k
    return max(active_count_dist.keys())

print(f"\n    99%   percentile: {percentile_active(99)} active")
print(f"    99.9% percentile: {percentile_active(99.9)} active")
print(f"    99.99% percentile: {percentile_active(99.99)} active")
print(f"    Max:              {max(active_count_dist.keys())} active")

# (2) TOP_N=4 충분한가: 픽셀 단위 overlap > N 인 픽셀 비율
total_px = sum(pixel_overlap_dist.values())
px_exceed = sum(c for k, c in pixel_overlap_dist.items() if k > TOP_N)
px_exceed_pct = 100 * px_exceed / max(total_px, 1)
covered_pct = 100 - px_exceed_pct
print(f"\n[2] pixel당 bbox > {TOP_N} 인 pixel: {px_exceed:,} / {total_px:,}  "
      f"({px_exceed_pct:.5f}% 손실, {covered_pct:.5f}% 커버)")
if covered_pct >= 99.99:
    print(f"    OK — TOP_N={TOP_N}으로 99.99% 이상 손실없이 표현")
elif covered_pct >= 99.9:
    print(f"    수용 — 손실 {px_exceed_pct:.3f}%")
else:
    print(f"    부족 — TOP_N 증가 검토")

# (3) frame 단위 overlap 분포
total_fmo = sum(max_overlap_hist.values())
print(f"\n[3] frame 단위 max overlap (한 frame에 가장 많이 겹친 pixel)")
cum = 0
for k in sorted(max_overlap_hist.keys()):
    c = max_overlap_hist[k]
    cum += c
    pct = c / max(total_fmo,1) * 100
    cum_pct = cum / max(total_fmo,1) * 100
    if pct >= 0.001 or k <= 10:
        print(f"    max_ov={k:3d}: {c:>10,}  ({pct:6.3f}%)  [cum {cum_pct:6.3f}%]")

# (4) 종합 권장
print(f"\n[4] 종합 결론")
overall_pixel_overlap = 100 * total_overlap_px / max(total_telop_px, 1)
print(f"    전체 telop pixel 중 overlap (≥2): {overall_pixel_overlap:.3f}%")
print(f"    K = {K_TARGET}, TOP_N = {TOP_N} 설정으로:")
print(f"      • Active slot 손실 {exceed_pct:.5f}%")
print(f"      • Pixel rank 손실 {px_exceed_pct:.5f}%")
total_loss = exceed_pct + px_exceed_pct
if total_loss < 0.05:
    print(f"    => Per-pixel Top-{TOP_N} 분류 (128 클래스) 진행 가능")
else:
    print(f"    => 손실 {total_loss:.3f}% — 정책 재검토")


In [ ]:
# %% 셀 7: 전체 데이터 per-pixel rank usage 분포 + focal loss alpha 추천
# 셀 2의 pixel_overlap_dist + n_total_frames 를 재활용 (추가 계산 없이 derive)

TOP_N_VAL = 4   # train_3와 동일

n_all_pixels = n_total_frames * GRID_H * GRID_W
n_bg_pixels  = n_all_pixels - total_telop_px

# rank usage 0..TOP_N_VAL 분포 (각 pixel에서 0이 아닌 rank slot 수)
usage = {0: n_bg_pixels}
for k in range(1, TOP_N_VAL):
    usage[k] = pixel_overlap_dist.get(k, 0)
usage[TOP_N_VAL] = sum(c for k, c in pixel_overlap_dist.items() if k >= TOP_N_VAL)

print(f"전체 픽셀: {n_all_pixels:,}")
print(f"  ({n_total_frames:,} frames × {GRID_H}×{GRID_W})")
print()
print(f"Per-pixel rank usage 분포 (TOP_N={TOP_N_VAL}):")
print(f"{'rank 사용수':>15} {'pixel':>18} {'비율':>12}")
print("─" * 60)
for n in range(TOP_N_VAL + 1):
    c = usage[n]
    pct = c / max(n_all_pixels, 1) * 100
    bar = "█" * min(int(pct / 2), 50)
    label = "BG (0 slot)" if n == 0 else f"{n} slot{'s' if n > 1 else ''}"
    if n == TOP_N_VAL:
        label += "+"
    print(f"  {label:>15}: {c:>15,}  ({pct:8.5f}%)  {bar}")

# ── Rank entry 수준 분포 (focal alpha 산정용) ──
# 각 pixel은 TOP_N_VAL개 rank entry를 가짐.
# k개 bbox 덮인 pixel의 rank entry = [slot×min(k,TOP_N), BG×(TOP_N-min(k,TOP_N))]
slot_rank = 0
bg_rank = 0
for n in range(TOP_N_VAL + 1):
    n_cover = min(n, TOP_N_VAL)
    slot_rank += usage[n] * n_cover
    bg_rank   += usage[n] * (TOP_N_VAL - n_cover)

total_rank = slot_rank + bg_rank
prior_slot = slot_rank / max(total_rank, 1)
prior_bg   = bg_rank / max(total_rank, 1)

print()
print(f"Rank entry 분포 (per-pixel TOP_N={TOP_N_VAL} 자리 기준):")
print(f"  Slot rank entry: {slot_rank:>15,}  ({prior_slot*100:8.5f}%)")
print(f"  BG rank entry:   {bg_rank:>15,}  ({prior_bg*100:8.5f}%)")

# ── Focal loss alpha 추천 ──
# Convention: rare class(slot)에 큰 alpha 부여
alpha_slot = 1.0 - prior_slot
alpha_bg   = prior_slot

print()
print("=" * 60)
print(f"[Focal loss alpha 추천] (γ=2 권장)")
print("=" * 60)
print(f"  FOCAL_ALPHA_SLOT = {alpha_slot:.6f}")
print(f"  FOCAL_ALPHA_BG   = {alpha_bg:.6f}")
print()
print(f"  → train_3.ipynb 셀 4 상수로 복사:")
print(f"     FOCAL_ALPHA_SLOT = {alpha_slot:.6f}")
print(f"     FOCAL_ALPHA_BG   = {alpha_bg:.6f}")
